In [ ]:
!pip -q install pystac-client planetary-computer rasterio geopandas shapely affine

In [ ]:
import numpy as np, pystac_client, planetary_computer, rasterio, geopandas as gpd
from rasterio.warp import transform_bounds, transform as coord_transform
from rasterio.windows import from_bounds, transform as window_transform
from rasterio.transform import xy as transform_xy
from affine import Affine
from scipy.ndimage import median_filter, uniform_filter, label, find_objects
from shapely.geometry import Point, box

SAHNE_ID = "S1A_IW_GRDH_1SDV_20230811T160002_20230811T160027_049830_05FE26_rtc"
sekil = (1000, 1000)

def v20(img):
    return img > uniform_filter(img, size=15) + 4.0

def v21(img):
    bg = (uniform_filter(img, size=15) * 225 - uniform_filter(img, size=5) * 25) / 200
    return img > bg + 4.0

katalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                    modifier=planetary_computer.sign_inplace)
kara = gpd.clip(gpd.read_file("https://naciscdn.org/naturalearth/10m/physical/ne_10m_land.zip"),
                box(28.85, 40.90, 29.15, 41.10))

for alan, bbox in [("Ahirkapi", [28.85, 40.85, 29.05, 40.95]),
                   ("Bosphorus", [28.90, 40.95, 29.10, 41.05])]:
    item = next(i for i in katalog.search(collections=["sentinel-1-rtc"], bbox=bbox,
                                          datetime="2023-08-01/2023-08-15").items() if i.id == SAHNE_ID)
    with rasterio.open(item.assets["vh"].href) as src:
        pencere = from_bounds(*transform_bounds("EPSG:4326", src.crs, *bbox), src.transform)
        ham = src.read(1, window=pencere, out_shape=sekil).astype(np.float32)
        src_transform, src_crs = src.transform, src.crs
    ham = np.nan_to_num(ham, nan=1e-6); ham[ham <= 0] = 1e-6
    vh_temiz = median_filter(10 * np.log10(ham), size=3)
    T = window_transform(pencere, src_transform) * Affine.scale(pencere.width / sekil[1], pencere.height / sekil[0])
    for surum, dedektor in [("v2.0", v20), ("v2.1", v21)]:
        kutular = [k for k in find_objects(label(dedektor(vh_temiz))[0])
                   if (k[0].stop - k[0].start) * (k[1].stop - k[1].start) >= 2]
        def karada_mi(k):
            x, y = transform_xy(T, (k[0].start + k[0].stop) / 2, (k[1].start + k[1].stop) / 2)
            lon, lat = coord_transform(src_crs, "EPSG:4326", [x], [y])
            return kara.contains(Point(lon[0], lat[0])).any()
        denizde = [k for k in kutular if not karada_mi(k)]
        print(f"RESULT {alan:<10} {surum} | raw: {len(kutular):>5} | removed as land: {len(kutular) - len(denizde):>5} | remaining: {len(denizde):>4}")
